# The Battle Class
This class is for making a Battle object.  A battle object takes a log (in the form of a string) as input.  From that string, it will gather the information of usernames, pre-battle ratings, post-battle ratings (if the match was rated), pokemon on each team, battle start and end time, and winner.  It has a nice string representation.

Next goal: allow it to take a .json file(name) as input instead of the raw log?

In [125]:
import json

class Battle:
    def __init__(self,file_name):
        with open("replays/gen9-randombattle/" + file_name,"r") as battle_json:
            data = json.load(battle_json)
        self.log = data["log"]
        self.player_names = ["",""]
        self.old_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings at the start of the match
        self.new_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings after the end of the match
        self.winner = ""
        self.loser = ""
        self.teams = {} # a dictionary where the keys are player usernames and the values are sets containing the pokemon names
        self.time_list = [] # for keeping track of all move times, gets thrown away now.  Could be used to check for time-out forfeits?
        self.start_time = 0
        self.end_time = 0
        self.rated = (data["rating"] != None)
        self.log_parser(self.log)

    def __repr__(self):
        to_return = ""
        if self.rated:
            to_return += f"This was a battle between {self.player_names[0]} (pre-match rating {self.old_player_ratings[self.player_names[0]]}) and {self.player_names[1]} (pre-match rating {self.old_player_ratings[self.player_names[1]]}).\n"
        else:
            to_return += f"This was a battle between {self.player_names[0]} and {self.player_names[1]}.\n"
        for index in range(2):
            to_return += f"{self.player_names[index]}'s team was {self.teams[self.player_names[index]]}.\n"
        to_return += f"{self.winner} won!\n"
        if self.rated:
            to_return += f"{self.winner}'s rating increased to {self.new_player_ratings[self.winner]}.\n"
            to_return += f"{self.loser}'s rating fell to {self.new_player_ratings[self.loser]}."
        else:
            to_return += "This was an unrated match, so no one's rating changed."
        return to_return

    def log_parser(self,log):
        # This loop generates player information based on the first (this is important) occurence of the substring "|player|"
        # In particular, it sets the Battle object's player_names and old_player_ratings attributes.
        for player_index in range(2):
            if player_index == 0:
                i = log.index("|player|p1|")
            else:
                i = log.index("|player|p2|")
            j = i+11
            # j is now the index of the first character in the player's username
            while log[j] != "|":
                self.player_names[player_index] += log[j]
                j += 1
            # initializes the teams as empty sets
            self.teams[self.player_names[player_index]] = set()
            # if the match is rated, gets the pre-battle ratings for each player
            if self.rated:
                # grabs the initial ratings of each player; tricky because you want to grab the last 3 or 4 characters of the line depending on 
                # whether the player's rating is 3 or 4 digits
                while log[j] != "\n":
                    j += 1
                # j is now the index of the \n character at the end of the line
                k = 0
                while log[j - k] != "|":
                    k = k+1
                # j - k is now the index of the | before the player's rating
                self.old_player_ratings[self.player_names[player_index]] = int(log[j-k+1:j])

        # This loop generates battle information by searching for *all* instances of "|t:|", "|switch|", "|win|", and " for winning)"
        # In particular, it sets the Battle object's new_player_ratings, winner, loser, teams, time_list, and rated attributes.
        for i in range(len(log)):
            # generates time list
            if log.startswith("|t:|",i):
                j = i + 4
                time_string = ""
                while log[j] != "\n":
                    time_string += log[j]
                    j += 1
                self.time_list.append(int(time_string))
            # generate team list
            if log.startswith("|switch|",i):
                pokemon_name = ""
                j = i+13
                while log[j] != "|":
                    pokemon_name += log[j]
                    j += 1
                if log.startswith("p1a",i+8):
                    player_name = self.player_names[0]
                else:
                    player_name = self.player_names[1]
                self.teams[player_name].add(pokemon_name)
            # identifies winner and loser
            if log.startswith("|win|",i):
                j = i + 5
                while log[j] != "\n":
                    self.winner += log[j]
                    j += 1
                winner_index = self.player_names.index(self.winner)
                self.loser = self.player_names[1-winner_index]
            if self.rated:
                if log.startswith(" for winning)",i):
                    delta_rating = int(log[i-2:i]) # assumes that the change in rating will be two digits
                    self.new_player_ratings[self.winner] = self.old_player_ratings[self.winner] + delta_rating
                    self.new_player_ratings[self.loser] = self.old_player_ratings[self.loser] - delta_rating
        # We separate start and end time for ease of access
        self.start_time = self.time_list[0]
        self.end_time = self.time_list[-1]

## Examples
Here, you can see how the Battle class functions on a couple of logs that I grabbed from Showdown on June 15.  It's pretty straightforward

In [126]:
battle1 = Battle("gen9randombattle-2631360263.json")
battle1

This was a battle between LaxMD (pre-match rating 1787) and N.TdaRajada (pre-match rating 1837).
LaxMD's team was {'Delphox', 'Indeedee', 'Wugtrio', 'Victreebel', 'Skarmory', 'Tauros'}.
N.TdaRajada's team was {'Snorlax', 'Lokix', 'Mandibuzz', 'Wigglytuff', 'Mienshao', 'Torterra'}.
N.TdaRajada won!
N.TdaRajada's rating increased to 1854.
LaxMD's rating fell to 1770.

In [127]:
battle2 = Battle("gen9randombattle-2631363920.json")
battle2

This was a battle between kaisarian and Flamesenpai557.
kaisarian's team was {'Breloom', 'Ambipom', 'Overqwil', 'Mimikyu', 'Brute Bonnet', 'Pincurchin'}.
Flamesenpai557's team was {'Breloom', 'Gurdurr', 'Kilowattrel', 'Arceus', 'Slaking', 'Dugtrio'}.
kaisarian won!
This was an unrated match, so no one's rating changed.

In [130]:
battle3 = Battle("gen9randombattle-2631365384.json")
battle3

This was a battle between rgrgreger (pre-match rating 1895) and Ismusicalschool (pre-match rating 1946).
rgrgreger's team was {'Donphan', 'Flareon', 'Electrode', 'Primarina', 'Sudowoodo'}.
Ismusicalschool's team was {'Houndoom', 'Slowking', 'Iron Treads', 'Lugia', 'Hippowdon', 'Luvdisc'}.
rgrgreger won!
rgrgreger's rating increased to 1918.
Ismusicalschool's rating fell to 1923.

In [131]:
battle4 = Battle("gen9randombattle-2631511033.json")
battle4

This was a battle between MMF6G and MandoWarrior.
MMF6G's team was {'Solgaleo', 'Drifblim', 'Gyarados', 'Conkeldurr', 'Torterra', 'Rampardos'}.
MandoWarrior's team was {'Smeargle', 'Golduck', 'Ting-Lu', 'Tinkaton', 'Whimsicott', 'Dondozo'}.
MandoWarrior won!
This was an unrated match, so no one's rating changed.

# Sandbox

Below is what I was using to test.  You should probably ignore this stuff unless you want to run some tests of your own without needing to type "self" everywhere...

In [118]:
log = "|j|☆kaisarian\n|j|☆Flamesenpai557\n|t:|1781375954\n|gametype|singles\n|player|p1|kaisarian|lucas-gen4pt|\n|player|p2|Flamesenpai557|101|\n|gen|9\n|tier|[Gen 9] Random Battle\n|rule|Species Clause: Limit one of each Pokémon\n|rule|HP Percentage Mod: HP is shown in percentages\n|rule|Sleep Clause Mod: Limit one foe put to sleep\n|rule|Illusion Level Mod: Illusion disguises the Pokémon's true level\n|\n|t:|1781375954\n|teamsize|p1|6\n|teamsize|p2|6\n|start\n|switch|p1a: Mimikyu|Mimikyu, L79, F|216/216\n|switch|p2a: Kilowattrel|Kilowattrel, L83, M|252/252\n|turn|1\n|l|☆Flamesenpai557\n|player|p2|\n|j|☆Flamesenpai557\n|player|p2|Flamesenpai557|101|\n|\n|t:|1781376040\n|move|p2a: Kilowattrel|Hurricane|p1a: Mimikyu\n|-activate|p1a: Mimikyu|ability: Disguise\n|-damage|p1a: Mimikyu|216/216\n|detailschange|p1a: Mimikyu|Mimikyu-Busted, L79, F\n|-damage|p1a: Mimikyu|189/216|[from] pokemon: Mimikyu-Busted\n|move|p1a: Mimikyu|Swords Dance|p1a: Mimikyu\n|-boost|p1a: Mimikyu|atk|2\n|\n|upkeep\n|turn|2\n|\n|t:|1781376063\n|move|p2a: Kilowattrel|Hurricane|p1a: Mimikyu|[miss]\n|-miss|p2a: Kilowattrel|p1a: Mimikyu\n|move|p1a: Mimikyu|Swords Dance|p1a: Mimikyu\n|-boost|p1a: Mimikyu|atk|2\n|\n|upkeep\n|turn|3\n|\n|t:|1781376078\n|move|p2a: Kilowattrel|Hurricane|p1a: Mimikyu|[miss]\n|-miss|p2a: Kilowattrel|p1a: Mimikyu\n|move|p1a: Mimikyu|Swords Dance|p1a: Mimikyu\n|-boost|p1a: Mimikyu|atk|2\n|\n|upkeep\n|turn|4\n|\n|t:|1781376087\n|move|p2a: Kilowattrel|Hurricane|p1a: Mimikyu|[miss]\n|-miss|p2a: Kilowattrel|p1a: Mimikyu\n|move|p1a: Mimikyu|Play Rough|p2a: Kilowattrel\n|-damage|p2a: Kilowattrel|0 fnt\n|faint|p2a: Kilowattrel\n|-damage|p1a: Mimikyu|168/216|[from] item: Life Orb\n|\n|upkeep\n|\n|t:|1781376130\n|switch|p2a: Dugtrio|Dugtrio-Alola, L84, F|196/196\n|turn|5\n|\n|t:|1781376142\n|move|p2a: Dugtrio|Iron Head|p1a: Mimikyu\n|-supereffective|p1a: Mimikyu\n|-damage|p1a: Mimikyu|0 fnt\n|faint|p1a: Mimikyu\n|-damage|p2a: Dugtrio|177/196|[from] item: Life Orb\n|\n|upkeep\n|\n|t:|1781376156\n|switch|p1a: Brute Bonnet|Brute Bonnet, L80|309/309\n|turn|6\n|\n|t:|1781376214\n|move|p1a: Brute Bonnet|Sucker Punch|p2a: Dugtrio\n|-damage|p2a: Dugtrio|66/196\n|-ability|p2a: Dugtrio|Tangling Hair\n|-unboost|p1a: Brute Bonnet|spe|1\n|move|p2a: Dugtrio|Iron Head|p1a: Brute Bonnet\n|-damage|p1a: Brute Bonnet|200/309\n|-damage|p2a: Dugtrio|47/196|[from] item: Life Orb\n|\n|-heal|p1a: Brute Bonnet|219/309|[from] item: Leftovers\n|upkeep\n|turn|7\n|\n|t:|1781376235\n|move|p1a: Brute Bonnet|Sucker Punch||[still]\n|-fail|p1a: Brute Bonnet\n|move|p2a: Dugtrio|Swords Dance|p2a: Dugtrio\n|-boost|p2a: Dugtrio|atk|2\n|\n|-heal|p1a: Brute Bonnet|238/309|[from] item: Leftovers\n|upkeep\n|turn|8\n|\n|t:|1781376252\n|move|p2a: Dugtrio|Sucker Punch|p1a: Brute Bonnet\n|-resisted|p1a: Brute Bonnet\n|-damage|p1a: Brute Bonnet|180/309\n|-damage|p2a: Dugtrio|28/196|[from] item: Life Orb\n|move|p1a: Brute Bonnet|Sucker Punch||[still]\n|-fail|p1a: Brute Bonnet\n|\n|-heal|p1a: Brute Bonnet|199/309|[from] item: Leftovers\n|upkeep\n|turn|9\n|\n|t:|1781376264\n|move|p2a: Dugtrio|Sucker Punch|p1a: Brute Bonnet\n|-resisted|p1a: Brute Bonnet\n|-damage|p1a: Brute Bonnet|138/309\n|-damage|p2a: Dugtrio|9/196|[from] item: Life Orb\n|move|p1a: Brute Bonnet|Crunch|p2a: Dugtrio\n|-damage|p2a: Dugtrio|0 fnt\n|-ability|p2a: Dugtrio|Tangling Hair\n|-unboost|p1a: Brute Bonnet|spe|1\n|faint|p2a: Dugtrio\n|\n|-heal|p1a: Brute Bonnet|157/309|[from] item: Leftovers\n|upkeep\n|\n|t:|1781376287\n|switch|p2a: Gurdurr|Gurdurr, L85, F|283/283\n|turn|10\n|\n|t:|1781376294\n|-terastallize|p1a: Brute Bonnet|Fighting\n|move|p2a: Gurdurr|Mach Punch|p1a: Brute Bonnet\n|-damage|p1a: Brute Bonnet|109/309\n|move|p1a: Brute Bonnet|Spore|p2a: Gurdurr\n|-status|p2a: Gurdurr|slp|[from] move: Spore\n|\n|-heal|p1a: Brute Bonnet|128/309|[from] item: Leftovers\n|upkeep\n|turn|11\n|\n|t:|1781376308\n|cant|p2a: Gurdurr|slp\n|move|p1a: Brute Bonnet|Close Combat|p2a: Gurdurr\n|-damage|p2a: Gurdurr|180/283 slp\n|-unboost|p1a: Brute Bonnet|def|1\n|-unboost|p1a: Brute Bonnet|spd|1\n|\n|-heal|p1a: Brute Bonnet|147/309|[from] item: Leftovers\n|upkeep\n|turn|12\n|\n|t:|1781376317\n|cant|p2a: Gurdurr|slp\n|move|p1a: Brute Bonnet|Close Combat|p2a: Gurdurr\n|-damage|p2a: Gurdurr|87/283 slp\n|-unboost|p1a: Brute Bonnet|def|1\n|-unboost|p1a: Brute Bonnet|spd|1\n|\n|-heal|p1a: Brute Bonnet|166/309|[from] item: Leftovers\n|upkeep\n|turn|13\n|\n|t:|1781376327\n|-curestatus|p2a: Gurdurr|slp|[msg]\n|move|p2a: Gurdurr|Drain Punch|p1a: Brute Bonnet\n|-damage|p1a: Brute Bonnet|0 fnt\n|-heal|p2a: Gurdurr|170/283|[from] drain|[of] p1a: Brute Bonnet\n|faint|p1a: Brute Bonnet\n|-end|p1a: Brute Bonnet|Protosynthesis|[silent]\n|\n|upkeep\n|\n|t:|1781376355\n|switch|p1a: Ambipom|Ambipom, L84, F|263/263\n|turn|14\n|\n|t:|1781376366\n|move|p1a: Ambipom|Low Kick|p2a: Gurdurr\n|-damage|p2a: Gurdurr|105/283\n|move|p2a: Gurdurr|Drain Punch|p1a: Ambipom\n|-supereffective|p1a: Ambipom\n|-damage|p1a: Ambipom|29/263\n|-heal|p2a: Gurdurr|222/283|[from] drain|[of] p1a: Ambipom\n|\n|upkeep\n|turn|15\n|\n|t:|1781376378\n|move|p2a: Gurdurr|Mach Punch|p1a: Ambipom\n|-supereffective|p1a: Ambipom\n|-damage|p1a: Ambipom|0 fnt\n|faint|p1a: Ambipom\n|\n|upkeep\n|\n|t:|1781376387\n|switch|p1a: Breloom|Breloom, L83, M|235/235\n|turn|16\n|\n|t:|1781376400\n|move|p1a: Breloom|Rock Tomb|p2a: Gurdurr\n|-resisted|p2a: Gurdurr\n|-damage|p2a: Gurdurr|187/283\n|-unboost|p2a: Gurdurr|spe|1\n|-damage|p1a: Breloom|212/235|[from] item: Life Orb\n|move|p2a: Gurdurr|Knock Off|p1a: Breloom\n|-resisted|p1a: Breloom\n|-damage|p1a: Breloom|172/235\n|-enditem|p1a: Breloom|Life Orb|[from] move: Knock Off|[of] p2a: Gurdurr\n|\n|upkeep\n|turn|17\n|\n|t:|1781376435\n|move|p1a: Breloom|Swords Dance|p1a: Breloom\n|-boost|p1a: Breloom|atk|2\n|move|p2a: Gurdurr|Drain Punch|p1a: Breloom\n|-damage|p1a: Breloom|84/235\n|-heal|p2a: Gurdurr|231/283|[from] drain|[of] p1a: Breloom\n|\n|upkeep\n|turn|18\n|\n|t:|1781376453\n|move|p2a: Gurdurr|Mach Punch|p1a: Breloom\n|-damage|p1a: Breloom|33/235\n|move|p1a: Breloom|Bullet Seed|p2a: Gurdurr\n|-damage|p2a: Gurdurr|168/283\n|-damage|p2a: Gurdurr|96/283\n|-hitcount|p2a: Gurdurr|2\n|\n|upkeep\n|turn|19\n|\n|t:|1781376460\n|move|p1a: Breloom|Mach Punch|p2a: Gurdurr\n|-damage|p2a: Gurdurr|0 fnt\n|faint|p2a: Gurdurr\n|\n|upkeep\n|\n|t:|1781376471\n|switch|p2a: Breloom|Breloom, L83, F|235/235\n|turn|20\n|\n|t:|1781376480\n|move|p2a: Breloom|Mach Punch|p1a: Breloom\n|-damage|p1a: Breloom|0 fnt\n|faint|p1a: Breloom\n|-damage|p2a: Breloom|212/235|[from] item: Life Orb\n|\n|upkeep\n|\n|t:|1781376488\n|switch|p1a: Overqwil|Overqwil, L82, F|274/274\n|-ability|p1a: Overqwil|Intimidate|boost\n|-unboost|p2a: Breloom|atk|1\n|turn|21\n|\n|t:|1781376508\n|-terastallize|p2a: Breloom|Fighting\n|move|p1a: Overqwil|Gunk Shot|p2a: Breloom|[miss]\n|-miss|p1a: Overqwil|p2a: Breloom\n|move|p2a: Breloom|Swords Dance|p2a: Breloom\n|-boost|p2a: Breloom|atk|2\n|\n|upkeep\n|turn|22\n|\n|t:|1781376566\n|move|p1a: Overqwil|Gunk Shot|p2a: Breloom|[miss]\n|-miss|p1a: Overqwil|p2a: Breloom\n|move|p2a: Breloom|Rock Tomb|p1a: Overqwil\n|-damage|p1a: Overqwil|115/274\n|-unboost|p1a: Overqwil|spe|1\n|-damage|p2a: Breloom|189/235|[from] item: Life Orb\n|\n|upkeep\n|turn|23\n|\n|t:|1781376578\n|move|p2a: Breloom|Rock Tomb|p1a: Overqwil\n|-damage|p1a: Overqwil|0 fnt\n|faint|p1a: Overqwil\n|-damage|p2a: Breloom|166/235|[from] item: Life Orb\n|\n|upkeep\n|\n|t:|1781376585\n|switch|p1a: Pincurchin|Pincurchin, M|258/258\n|-fieldstart|move: Electric Terrain|[from] ability: Electric Surge|[of] p1a: Pincurchin\n|turn|24\n|\n|t:|1781376615\n|move|p2a: Breloom|Swords Dance|p2a: Breloom\n|-boost|p2a: Breloom|atk|2\n|move|p1a: Pincurchin|Thunderbolt|p2a: Breloom\n|-damage|p2a: Breloom|0 fnt\n|faint|p2a: Breloom\n|\n|upkeep\n|\n|t:|1781376636\n|switch|p2a: Slaking|Slaking, L84, M|389/389\n|turn|25\n|\n|t:|1781376643\n|move|p2a: Slaking|Giga Impact|p1a: Pincurchin\n|-damage|p1a: Pincurchin|80/258\n|-mustrecharge|p2a: Slaking\n|move|p1a: Pincurchin|Toxic Spikes|p2a: Slaking\n|-sidestart|p2: Flamesenpai557|move: Toxic Spikes\n|\n|-heal|p1a: Pincurchin|96/258|[from] item: Leftovers\n|upkeep\n|turn|26\n|\n|t:|1781376652\n|cant|p2a: Slaking|recharge\n|move|p1a: Pincurchin|Thunderbolt|p2a: Slaking\n|-crit|p2a: Slaking\n|-damage|p2a: Slaking|55/389\n|\n|-heal|p1a: Pincurchin|112/258|[from] item: Leftovers\n|upkeep\n|turn|27\n|\n|t:|1781376666\n|switch|p2a: Arceus|Arceus-Ghost, L69|280/280\n|-status|p2a: Arceus|psn\n|move|p1a: Pincurchin|Recover|p1a: Pincurchin\n|-heal|p1a: Pincurchin|241/258\n|\n|-heal|p1a: Pincurchin|257/258|[from] item: Leftovers\n|-damage|p2a: Arceus|245/280 psn|[from] psn\n|upkeep\n|turn|28\n|\n|t:|1781376684\n|move|p2a: Arceus|Judgment|p1a: Pincurchin\n|-damage|p1a: Pincurchin|175/258\n|move|p1a: Pincurchin|Thunderbolt|p2a: Arceus\n|-damage|p2a: Arceus|82/280 psn\n|\n|-heal|p1a: Pincurchin|191/258|[from] item: Leftovers\n|-damage|p2a: Arceus|47/280 psn|[from] psn\n|-fieldend|move: Electric Terrain\n|upkeep\n|turn|29\n|j| KidGold2108\n|\n|t:|1781376710\n|move|p2a: Arceus|Focus Blast|p1a: Pincurchin|[miss]\n|-miss|p2a: Arceus|p1a: Pincurchin\n|move|p1a: Pincurchin|Thunderbolt|p2a: Arceus\n|-damage|p2a: Arceus|0 fnt\n|faint|p2a: Arceus\n|\n|-heal|p1a: Pincurchin|207/258|[from] item: Leftovers\n|upkeep\n|\n|t:|1781376720\n|switch|p2a: Slaking|Slaking, L84, M|55/389\n|-status|p2a: Slaking|psn\n|turn|30\n|\n|t:|1781376744\n|move|p2a: Slaking|Earthquake|p1a: Pincurchin\n|-supereffective|p1a: Pincurchin\n|-damage|p1a: Pincurchin|49/258\n|move|p1a: Pincurchin|Recover|p1a: Pincurchin\n|-heal|p1a: Pincurchin|178/258\n|\n|-heal|p1a: Pincurchin|194/258|[from] item: Leftovers\n|-damage|p2a: Slaking|7/389 psn|[from] psn\n|upkeep\n|turn|31\n|\n|t:|1781376754\n|cant|p2a: Slaking|ability: Truant\n|move|p1a: Pincurchin|Recover|p1a: Pincurchin\n|-heal|p1a: Pincurchin|258/258\n|\n|-damage|p2a: Slaking|0 fnt|[from] psn\n|faint|p2a: Slaking\n|\n|win|kaisarian\n"
player_names = ["",""]
old_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings at the start of the match
new_player_ratings = {} # a dictionary where the keys are the player usernames and the values are the player ratings after the end of the match
winner = ""
loser = ""
teams = {} # a dictionary where the keys are player usernames and the values are sets containing the pokemon names
time_list = [] # for keeping track of all move times, gets thrown away now.  Could be used to check for time-out forfeits?
start_time = 0
end_time = 0


# generates player usernames, initializes the teams dict with an empty set for each player, sets the old_player_ratings
for player_index in range(2):
    if player_index == 0:
        i = log.index("|player|p1|")
    else:
        i = log.index("|player|p2|")
    # i is now the index of the first pipe of the first instance the player joins
    j = i+11
    # j is now the index of the first character in the player's username
    while log[j] != "|":
        player_names[player_index] += log[j]
        j += 1
    # initializes the teams as empty sets
    teams[player_names[player_index]] = set()
    # grabs the initial ratings of each player; tricky because you want to grab the last 3 or 4 characters of the line depending on 
    # whether the player's rating is 3 or 4 digits
    while log[j] != "\n":
        j += 1
    # j is now the index of the \n character at the end of the line
    k = 0
    while log[j - k] != "|":
        k = k+1
    # j - k is now the index of the | before the player's rating
    old_player_ratings[player_names[player_index]] = int(log[j-k+1:j])

for i in range(len(log)):
    # generates time list
    if log.startswith("|t:",i):
        j = i + 4
        time_string = ""
        while log[j] != "\n":
            time_string += log[j]
            j += 1
        time_list.append(int(time_string))
    # generate team list
    if log.startswith("|switch|",i):
        pokemon_name = ""
        j = i+13
        while log[j] != "|":
            pokemon_name += log[j]
            j += 1
        if log.startswith("p1a",i+8):
            player_name = player_names[0]
        else:
            player_name = player_names[1]
        teams[player_name].add(pokemon_name)
    # identifies winner and loser
    if log.startswith("|win|",i):
        j = i + 5
        while log[j] != "\n":
            winner += log[j]
            j += 1
        winner_index = player_names.index(winner)
        loser = player_names[1-winner_index]
    if log.startswith(" for winning)",i):
        delta_rating = int(log[i-2:i]) # assumes that the change in rating will be two digits
        new_player_ratings[winner] = old_player_ratings[winner] + delta_rating
        new_player_ratings[loser] = old_player_ratings[loser] - delta_rating
    

start_time = time_list[0]
end_time = time_list[-1]

for index in range(2):
    print(f"{player_names[index]}'s old rating:", old_player_ratings[player_names[index]])
    print(f"{player_names[index]}'s team:", teams[player_names[index]])
print("winner:", winner)
try:
    for index in range(2):
        print(f"{player_names[index]} new rating:", new_player_ratings[player_names[index]])
except KeyError:
    print("This was not a rated match")

print("start time:", start_time)
print("end time:", end_time)

ValueError: invalid literal for int() with base 10: ''

In [85]:
line = "|player|p1|xxzfn|170|2221\n"
j = 11
# grabs the index of the player: 1 if p2, 0 else; this way, we avoid the assumption that player 1's name is printed first
while line[j] != "|":
    j += 1
# grabs the initial ratings of each player; tricky because you want to grab the last 3 or 4 characters of the line depending on 
# whether the player's rating is 3 or 4 digits
while line[j] != "\n":
    j += 1
    print(f"line[{j}] = {line[j]}")
# j is now the index of the \n character at the end of the line
k = 0
while line[j - k] != "|":
    k = k+1
    print(f"k = {k}")
# j - k is now the index of the | before the player's rating
# old_player_ratings[player_names[player_index]] = int(log[j-k+1:j-1])
print("rating:",line[j-k+1:j])


line[17] = 1
line[18] = 7
line[19] = 0
line[20] = |
line[21] = 2
line[22] = 2
line[23] = 2
line[24] = 1
line[25] = 

k = 1
k = 2
k = 3
k = 4
k = 5
rating: 2221


In [43]:
len(line)

26

In [121]:
with open("replays/gen9-randombattle/gen9randombattle-2631363920.json","r") as battle_json:
    data = json.load(battle_json)

data["rating"] == None

True